In [2]:
import pandas as pd
import numpy as np
from google.colab import files


uploaded = files.upload()


df = pd.read_csv('diabetic_data.csv', na_values=['?'])

print("Orijinal Veri Seti Boyutu (Satır, Sütun):", df.shape)
df.head()

Saving diabetic_data.csv to diabetic_data.csv


/tmp/ipykernel_738/3920863910.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('diabetic_data.csv', na_values=['?'])


Orijinal Veri Seti Boyutu (Satır, Sütun): (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:

columns_to_drop = ['weight', 'payer_code', 'medical_specialty']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])


for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())


for col in df.select_dtypes(include=['object']).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Eksik veriler başarıyla temizlendi ve dolduruldu.")
print("Kalan toplam eksik veri sayısı:", df.isnull().sum().sum())

Eksik veriler başarıyla temizlendi ve dolduruldu.
Kalan toplam eksik veri sayısı: 0


In [4]:

categorical_cols = ['race', 'gender', 'admission_type_id']
existing_cat_cols = [col for col in categorical_cols if col in df.columns]

df = pd.get_dummies(df, columns=existing_cat_cols, drop_first=True)

print("Kategorik dönüşüm tamamlandı. Yeni sütun sayısı:", df.shape[1])

Kategorik dönüşüm tamamlandı. Yeni sütun sayısı: 57


In [6]:
from sklearn.preprocessing import StandardScaler


numeric_cols = df.select_dtypes(include=[np.number]).columns

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("Özellik ölçeklendirme (Standardizasyon) başarıyla tamamlandı!")
df[numeric_cols].head()

Özellik ölçeklendirme (Standardizasyon) başarıyla tamamlandı!


,encounter_id,patient_nbr,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
0,-1.587330,-1.191545,4.031022,-1.169873,-1.137649,-0.106226,-0.785398,-1.848268,-0.291461,-0.21262,-0.503276,-3.321596
1,-1.608075,0.033564,-0.514312,0.306482,-0.467653,0.809860,-0.785398,0.243390,-0.291461,-0.21262,-0.503276,0.815784
2,-1.608901,0.819654,-0.514312,0.306482,-0.802651,-1.633035,2.145781,-0.371804,1.286748,-0.21262,0.288579,-0.735733
3,-1.604653,0.726480,-0.514312,0.306482,-0.802651,0.046455,-0.199162,-0.002688,-0.291461,-0.21262,-0.503276,-0.218561
4,-1.609366,-0.305227,-0.514312,0.306482,-1.137649,0.402711,-0.785398,-0.986997,-0.291461,-0.21262,-0.503276,-1.252906


In [7]:

if 'time_in_hospital' in df.columns:
    bins = [-1, 2, 5, 20]
    labels = [0, 1, 2]
    df['Stay_Category'] = pd.cut(df['time_in_hospital'], bins=bins, labels=labels)
    df['Stay_Category'] = df['Stay_Category'].fillna(0).astype(int)

df.to_csv('cleaned_diabetes_final.csv', index=False)

print("Tüm ön işleme adımları başarıyla tamamlandı!")
print("Kaydedilen final veri boyutu:", df.shape)

Tüm ön işleme adımları başarıyla tamamlandı!
Kaydedilen final veri boyutu: (101766, 58)
